# Full-document chunking corpus analysis

Quick corpus checks before building the structure-aware chunker.

In [ ]:
%cd ..

from collections import Counter
from pathlib import Path
import re

import numpy as np
import pandas as pd

from src.io_utils import iter_jsonl
from src.paths import CORPUS_PATH


In [ ]:
blank_line_re = re.compile(r"\n\s*\n")

section_patterns = {
    "abstract": re.compile(r"(?m)^\s*Abstract\s*$"),
    "introduction": re.compile(
        r"(?im)^\s*(?:\d+(?:\.\d+)*)?[.)]?\s*Introduction\s*$"
    ),
}

reference_patterns = {
    "figure_reference": re.compile(
        r"\b(?:Figure|Fig\.?)(?:\s+|~)?\d+[A-Za-z]?\b",
        re.IGNORECASE,
    ),
    "table_reference": re.compile(
        r"\bTable(?: |\u00a0)\d+\b",
        re.IGNORECASE,
    ),
}

markup_patterns = {
    "markdown_table_strict": re.compile(r"(?m)^\|.+\|\s*$"),
    "markdown_table_loose": re.compile(r"(?m)^\s*\|?\s*:?-{3,}.*\|.*$"),
    "latex_tabular": re.compile(r"\\begin\{tabular\}|\\end\{tabular\}|\\tabular"),
    "html_table": re.compile(r"<table\b|</table>", re.IGNORECASE),
    "image_marker": re.compile(r"!\[|<img\b|\\includegraphics", re.IGNORECASE),
}


def word_count(text: str) -> int:
    return len(text.split())


def split_blocks(text: str) -> list[str]:
    stripped = text.strip()
    if not stripped:
        return []
    return [block.strip() for block in blank_line_re.split(stripped) if block.strip()]


def summarize_distribution(values: np.ndarray) -> dict[str, float]:
    return {
        "median": float(np.quantile(values, 0.5)),
        "p90": float(np.quantile(values, 0.9)),
        "p99": float(np.quantile(values, 0.99)),
        "max": float(np.max(values)),
    }


def pct(count: int, total: int) -> float:
    return count / total * 100 if total else float("nan")


def analyze_corpus(path: Path) -> dict[str, object]:
    first_row_keys: list[str] | None = None
    doc_lengths: list[int] = []
    block_counts: list[int] = []
    block_lengths: list[int] = []

    counts: Counter[str] = Counter()
    short_block_total = 0
    docs_with_many_short_blocks = 0

    for row in iter_jsonl(path):
        if first_row_keys is None:
            first_row_keys = sorted(str(key) for key in row.keys())

        counts["documents"] += 1
        text = str(row["default_text"])
        doc_lengths.append(word_count(text))

        for name, pattern in section_patterns.items():
            if pattern.search(text):
                counts[f"{name}_docs"] += 1

        for name, pattern in reference_patterns.items():
            if pattern.search(text):
                counts[f"{name}_docs"] += 1

        for name, pattern in markup_patterns.items():
            if pattern.search(text):
                counts[f"{name}_docs"] += 1

        blocks = split_blocks(text)
        block_counts.append(len(blocks))

        short_blocks_in_doc = 0
        for block in blocks:
            block_length = word_count(block)
            block_lengths.append(block_length)
            if block_length <= 3:
                short_block_total += 1
                short_blocks_in_doc += 1

        if short_blocks_in_doc >= 10:
            docs_with_many_short_blocks += 1

    doc_length_array = np.asarray(doc_lengths)
    block_count_array = np.asarray(block_counts)
    block_length_array = np.asarray(block_lengths)
    block_sample = pd.Series(block_length_array).sample(
        n=min(200_000, len(block_length_array)),
        random_state=42,
    ).to_numpy()

    return {
        "schema_keys": first_row_keys or [],
        "doc_count": counts["documents"],
        "doc_lengths": doc_length_array,
        "block_counts": block_count_array,
        "block_lengths": block_length_array,
        "block_sample": block_sample,
        "doc_length_summary": summarize_distribution(doc_length_array),
        "block_count_summary": summarize_distribution(block_count_array),
        "block_length_summary": summarize_distribution(block_length_array),
        "block_sample_summary": summarize_distribution(block_sample),
        "short_block_total": short_block_total,
        "docs_with_many_short_blocks": docs_with_many_short_blocks,
        "section_counts": {
            "abstract": counts["abstract_docs"],
            "introduction": counts["introduction_docs"],
        },
        "reference_counts": {
            "figure_reference": counts["figure_reference_docs"],
            "table_reference": counts["table_reference_docs"],
        },
        "markup_counts": {
            "markdown_table_strict": counts["markdown_table_strict_docs"],
            "markdown_table_loose": counts["markdown_table_loose_docs"],
            "latex_tabular": counts["latex_tabular_docs"],
            "html_table": counts["html_table_docs"],
            "image_marker": counts["image_marker_docs"],
        },
    }


In [ ]:
analysis = analyze_corpus(CORPUS_PATH)
analysis["doc_count"], analysis["schema_keys"]


In [ ]:
corpus_snapshot_df = pd.DataFrame(
    [
        {
            "path": str(CORPUS_PATH),
            "size_gib": round(CORPUS_PATH.stat().st_size / (1024 ** 3), 2),
            "documents": analysis["doc_count"],
            "schema": ", ".join(analysis["schema_keys"]),
        }
    ]
)

corpus_snapshot_df


## Length and block structure

In [ ]:
high_level_length_df = pd.DataFrame(
    [
        {
            "measure": "Document length (words)",
            **analysis["doc_length_summary"],
        },
        {
            "measure": "Paragraph-like blocks per doc",
            **analysis["block_count_summary"],
        },
    ]
)

high_level_length_df


In [ ]:
block_length_df = pd.DataFrame(
    [
        {
            "view": "All paragraph-like blocks",
            **analysis["block_length_summary"],
        },
        {
            "view": "200k block sample",
            **analysis["block_sample_summary"],
        },
    ]
)

block_length_df


In [ ]:
block_lengths = analysis["block_lengths"]

long_block_threshold_df = pd.DataFrame(
    [
        {"threshold": ">200 words", "pct_of_blocks": pct(int((block_lengths > 200).sum()), len(block_lengths))},
        {"threshold": ">300 words", "pct_of_blocks": pct(int((block_lengths > 300).sum()), len(block_lengths))},
        {"threshold": ">500 words", "pct_of_blocks": pct(int((block_lengths > 500).sum()), len(block_lengths))},
        {"threshold": ">800 words", "pct_of_blocks": pct(int((block_lengths > 800).sum()), len(block_lengths))},
    ]
)

long_block_threshold_df


## Section markers and noise

In [ ]:
section_and_noise_df = pd.DataFrame(
    [
        {
            "signal": "Docs with an Abstract heading",
            "docs": analysis["section_counts"]["abstract"],
            "pct_of_docs": pct(analysis["section_counts"]["abstract"], analysis["doc_count"]),
        },
        {
            "signal": "Docs with an Introduction heading",
            "docs": analysis["section_counts"]["introduction"],
            "pct_of_docs": pct(analysis["section_counts"]["introduction"], analysis["doc_count"]),
        },
        {
            "signal": "Short blocks (<=3 words)",
            "docs": analysis["short_block_total"],
            "pct_of_docs": pct(analysis["short_block_total"], len(analysis["block_lengths"])),
        },
        {
            "signal": "Docs with >=10 short blocks",
            "docs": analysis["docs_with_many_short_blocks"],
            "pct_of_docs": pct(analysis["docs_with_many_short_blocks"], analysis["doc_count"]),
        },
    ]
)

section_and_noise_df


## Tables, figures, and richer markup

In [ ]:
reference_df = pd.DataFrame(
    [
        {
            "signal": "Figure references",
            "docs": analysis["reference_counts"]["figure_reference"],
            "pct_of_docs": pct(analysis["reference_counts"]["figure_reference"], analysis["doc_count"]),
        },
        {
            "signal": "Table references",
            "docs": analysis["reference_counts"]["table_reference"],
            "pct_of_docs": pct(analysis["reference_counts"]["table_reference"], analysis["doc_count"]),
        },
    ]
)

reference_df


In [ ]:
markup_df = pd.DataFrame(
    [
        {
            "signal": "Markdown-table-like (strict)",
            "docs": analysis["markup_counts"]["markdown_table_strict"],
            "pct_of_docs": pct(analysis["markup_counts"]["markdown_table_strict"], analysis["doc_count"]),
        },
        {
            "signal": "Markdown-table-like (loose)",
            "docs": analysis["markup_counts"]["markdown_table_loose"],
            "pct_of_docs": pct(analysis["markup_counts"]["markdown_table_loose"], analysis["doc_count"]),
        },
        {
            "signal": "LaTeX tabular markers",
            "docs": analysis["markup_counts"]["latex_tabular"],
            "pct_of_docs": pct(analysis["markup_counts"]["latex_tabular"], analysis["doc_count"]),
        },
        {
            "signal": "HTML table markers",
            "docs": analysis["markup_counts"]["html_table"],
            "pct_of_docs": pct(analysis["markup_counts"]["html_table"], analysis["doc_count"]),
        },
        {
            "signal": "Explicit image include markers",
            "docs": analysis["markup_counts"]["image_marker"],
            "pct_of_docs": pct(analysis["markup_counts"]["image_marker"], analysis["doc_count"]),
        },
    ]
)

markup_df


where that leaves us: split on blank lines, carry nearby headings as context, merge tiny blocks, and fall back to sentence windows for oversized blocks.